In [ ]:
!pip install -q datasets transformers accelerate evaluate jiwer tensorboard

In [ ]:
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
import torch

# Load Uzbek speech corpus
dataset = load_dataset("murodbek/uzbek-speech-corpus")
print(dataset)
print(dataset["train"][0])  # look at one example

In [ ]:
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, language="uz", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Force Uzbek language token during generation
model.generation_config.language = "uz"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
# Shuffle and take 2500 for training, use existing test split
train_dataset = dataset["train"].shuffle(seed=42).select(range(2500))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
def prepare_dataset(batch):
    # Resample audio to 16kHz (Whisper expects this)
    audio = batch["audio"]
    
    # Extract mel spectrogram features
    batch["input_features"] = processor.feature_extractor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Encode target text to label IDs
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

# Apply preprocessing
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names)

In [ ]:
import dataclasses
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Stack input features (mel spectrograms)
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels (target text token IDs)
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding token id with -100 so it's ignored in loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if it was added (Whisper doesn't need it at start)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Evaluate base model BEFORE training — this is your "before" number
from torch.utils.data import DataLoader

model.eval()
predictions = []
references = []

# Test on a small batch to get baseline WER
sample_eval = dataset["test"].shuffle(seed=42).select(range(50))

for sample in sample_eval:
    audio = sample["audio"]
    input_features = processor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"], 
        return_tensors="pt"
    ).input_features.to(model.device)
    
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    predictions.append(transcription)
    references.append(sample["sentence"])

baseline_wer = wer_metric.compute(predictions=predictions, references=references)
print(f"BASELINE WER (before fine-tuning): {baseline_wer:.4f}")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-uzbek",
    per_device_train_batch_size=8,       # reduce to 4 if OOM
    gradient_accumulation_steps=2,        # effective batch = 16
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,                       # ~1000 steps enough for 2500 samples
    eval_strategy="steps",
    eval_steps=250,
    save_steps=250,
    logging_steps=50,
    fp16=True,                            # half precision — saves VRAM on T4
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"],
    push_to_hub=False,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()

In [ ]:
# Same evaluation as Cell 8, but now model is fine-tuned
model.eval()
predictions_ft = []

for sample in sample_eval:
    audio = sample["audio"]
    input_features = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(model.device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    predictions_ft.append(transcription)

finetuned_wer = wer_metric.compute(predictions=predictions_ft, references=references)
print(f"BASELINE WER:    {baseline_wer:.4f}")
print(f"FINE-TUNED WER:  {finetuned_wer:.4f}")
print(f"IMPROVEMENT:     {(baseline_wer - finetuned_wer) / baseline_wer * 100:.1f}%")

In [ ]:
model.save_pretrained("./whisper-small-uzbek-finetuned")
processor.save_pretrained("./whisper-small-uzbek-finetuned")
print("Model saved!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save dataset to Drive
dataset.save_to_disk("/content/drive/MyDrive/uzbek_speech_corpus")

In [ ]:
!cp -r ./whisper-small-uzbek-finetuned /content/drive/MyDrive/whisper-small-uzbek-finetuned